# 04 · Phases, Lattices, Space Groups, and CIF Import

**Where this sits in PyTex.** Notebooks 01–03 were geometric and symbolic. Now we
attach that machinery to *real crystals*. A **`Phase`** bundles a **`Lattice`** (cell
geometry), a **`UnitCell`** (the atomic basis), a **space group**, and a
**`SymmetrySpec`** into the single object that every downstream diffraction and
texture workflow consumes. We build phases from PyTex's **pinned, hash-verified CIF
fixture corpus**, so the numbers in this tutorial are reproducible and provenance-backed.

## Learning goals

1. browse the fixture corpus and read each phase's crystallographic provenance;
2. load a `Phase` and inspect its lattice geometry — direct basis, **metric tensor**, and **cell volume**;
3. use the **reciprocal metric tensor** to compute interplanar **d-spacings**, checked against analytic values;
4. read the **atomic basis** and confirm site counts against the fixture's expected values;
5. round-trip a structure through **CIF** text.

## Theory: the metric tensor is the geometry

A lattice is defined by six parameters $(a,b,c,\alpha,\beta,\gamma)$. All lengths and
angles in the crystal follow from the **metric tensor**

$$
\mathbf{G} = \begin{pmatrix} \mathbf{a}\cdot\mathbf{a} & \mathbf{a}\cdot\mathbf{b} & \mathbf{a}\cdot\mathbf{c}\\ \mathbf{b}\cdot\mathbf{a} & \mathbf{b}\cdot\mathbf{b} & \mathbf{b}\cdot\mathbf{c}\\ \mathbf{c}\cdot\mathbf{a} & \mathbf{c}\cdot\mathbf{b} & \mathbf{c}\cdot\mathbf{c}\end{pmatrix},
\qquad
V = \sqrt{\det \mathbf{G}}.
$$

Its inverse is the **reciprocal metric tensor** $\mathbf{G}^{*}=\mathbf{G}^{-1}$, from
which the spacing of the $(hkl)$ planes is

$$
\frac{1}{d_{hkl}^{2}} = \begin{pmatrix} h & k & l\end{pmatrix}\,\mathbf{G}^{*}\,\begin{pmatrix} h\\ k\\ l\end{pmatrix}.
$$

This single relation replaces the seven system-specific $d$-spacing formulas found in
textbooks and is exact for triclinic through cubic alike. (De Graef & McHenry,
*Structure of Materials*, 2nd ed., 2012; International Tables Vol. A.)

In [ ]:
from __future__ import annotations

import warnings

import numpy as np

# The pinned fixture CIFs omit an explicit symmetry-operations loop, so pymatgen
# derives the group from the H-M symbol and warns. That is expected for these
# minimal prototypes; silence it so the tutorial output stays readable.
warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    Phase,
    ReferenceFrame,
    get_phase_fixture,
    list_phase_fixtures,
)

np.set_printoptions(precision=4, suppress=True)
crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))

## 1 · The fixture corpus and its provenance

Every fixture carries a `metadata` dictionary recording the source database, the
space group, the reference lattice parameters, and the *expected* site count — the
independent "known answer" against which import is validated. The corpus spans the
common structure prototypes (BCC, FCC, HCP, diamond-cubic, and a layered trigonal
compound).

In [ ]:
print(f"{'id':10} {'display name':22} {'system':10} {'space group':12} sites(conv)")
print("-" * 72)
for record in list_phase_fixtures():
    meta = record.metadata
    print(f"{record.fixture_id:10} {meta['display_name']:22} "
          f"{meta['crystal_system']:10} {meta['space_group_symbol']:12} "
          f"{meta['expected_conventional_cell_site_count']}")

## 2 · Loading a phase and reading its geometry

We load hexagonal-close-packed zirconium. The `Lattice` exposes the **direct basis**
(the Cartesian components of $\mathbf{a},\mathbf{b},\mathbf{c}$), the **metric
tensor**, and — via $V=\sqrt{\det\mathbf G}$ — the cell volume. For an HCP cell the
volume has the closed form $V = \tfrac{\sqrt3}{2}\,a^{2}c$, which we assert.

In [ ]:
zr = get_phase_fixture("zr_hcp").load_phase(crystal_frame=crystal)
lattice = zr.lattice
print(f"Zr HCP: a={lattice.a:.4f}  c={lattice.c:.4f} A   gamma={lattice.gamma_deg:.1f} deg")
print("\ndirect basis (columns a, b, c in Angstrom):\n", lattice.direct_basis().matrix)

G = lattice.metric_tensor()
volume = np.sqrt(np.linalg.det(G))
volume_hcp = np.sqrt(3.0) / 2.0 * lattice.a**2 * lattice.c
print("\nmetric tensor G:\n", G)
print(f"\ncell volume = sqrt(det G) = {volume:.4f} A^3")
print(f"analytic HCP volume  (sqrt3/2) a^2 c = {volume_hcp:.4f} A^3")
assert abs(volume - volume_hcp) < 1e-6

## 3 · d-spacings from the reciprocal metric tensor

Switching to FCC nickel (cubic, $a=3.52387$ Å) lets us check the reciprocal-metric
$d$-spacing against the cubic identity $d_{hkl}=a/\sqrt{h^2+k^2+l^2}$. The same
`reciprocal_metric_tensor` call would give the correct, non-obvious spacings for the
hexagonal or trigonal fixtures — that is the point of using the tensor form.

In [ ]:
ni = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
G_star = ni.lattice.reciprocal_metric_tensor()

def d_spacing(hkl):
    h = np.asarray(hkl, dtype=float)
    return 1.0 / np.sqrt(h @ G_star @ h)

a_ni = ni.lattice.a
print(f"{'plane':>7} | {'d (PyTex)':>10} | {'d = a/sqrt(N)':>13}")
for hkl in ([1, 1, 1], [2, 0, 0], [2, 2, 0], [3, 1, 1]):
    analytic = a_ni / np.sqrt(np.dot(hkl, hkl))
    print(f"{str(tuple(hkl)):>7} | {d_spacing(hkl):>10.5f} | {analytic:>13.5f}")
    assert abs(d_spacing(hkl) - analytic) < 1e-9

## 4 · The atomic basis

The `UnitCell` holds the `AtomicSite` list — species, fractional coordinates,
occupancy. The number of sites in the conventional cell is a structural fingerprint
(2 for BCC/HCP, 4 for FCC, 8 for diamond) and must match the value the fixture
declares as its known-good result. We verify this for every fixture: a mismatch would
mean the CIF import silently changed the structure.

In [ ]:
for record in list_phase_fixtures():
    phase = record.load_phase(crystal_frame=crystal)
    n_sites = len(phase.unit_cell.sites)
    expected = record.metadata["expected_conventional_cell_site_count"]
    flag = "OK" if n_sites == expected else "MISMATCH"
    print(f"{record.fixture_id:10} sites={n_sites}  expected={expected}  [{flag}]")
    assert n_sites == expected

# Peek at the diamond basis: two symmetry-distinct carbon positions expand to 8 sites.
diamond = get_phase_fixture("diamond").load_phase(crystal_frame=crystal)
print("\nDiamond sites (first three):")
for site in diamond.unit_cell.sites[:3]:
    print(f"  {site.species:2} at {np.round(site.fractional_coordinates, 4)}")

## 5 · Space group, symmetry, and CIF round-trip

The `Phase` also carries the space-group symbol/number and the point-group
`SymmetrySpec` (Notebook 03) consistent with it. Finally we demonstrate that a phase
survives a round-trip through CIF text — `read_cif_text` → `Phase.from_cif_string` —
returning the same space group and site count. This is the interoperability contract:
PyTex phases are portable crystallographic records, not opaque in-memory blobs.

In [ ]:
print(f"Ni: space group {ni.space_group_symbol} (No. {ni.space_group_number}), "
      f"point group {ni.symmetry.point_group}, symmetry order {ni.symmetry.order}")

fe_record = get_phase_fixture("fe_bcc")
cif_text = fe_record.read_cif_text()
fe_reloaded = Phase.from_cif_string(cif_text, crystal_frame=crystal)
print("\nCIF round-trip (alpha-Fe):")
print("  space group:", fe_reloaded.space_group_symbol,
      "| sites:", len(fe_reloaded.unit_cell.sites),
      "| point group:", fe_reloaded.symmetry.point_group)
assert fe_reloaded.space_group_symbol == fe_record.metadata["space_group_symbol"]

### Visualising the HCP basal plane

The metric tensor's off-diagonal $\mathbf{a}\cdot\mathbf{b}=a^2\cos120^\circ$ is what
makes the hexagonal cell non-orthogonal. Drawing the basal basis vectors $\mathbf{a}_1$
and $\mathbf{a}_2$ at $120^\circ$ makes the geometry behind the metric tensor visible.

In [ ]:
import matplotlib.pyplot as plt

basis = zr.lattice.direct_basis().matrix
a1, a2 = basis[:2, 0], basis[:2, 1]

fig, ax = plt.subplots(figsize=(4.4, 4.4))
for vec, label, color in ((a1, r"$\mathbf{a}_1$", "#d1495b"), (a2, r"$\mathbf{a}_2$", "#2e86ab")):
    ax.annotate("", xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.text(vec[0] * 1.1, vec[1] * 1.1, label, color=color, fontsize=13)
# unit-cell rhombus
cell = np.array([[0, 0], a1, a1 + a2, a2, [0, 0]])
ax.plot(cell[:, 0], cell[:, 1], color="0.6", lw=1, ls="--")
ax.set_aspect("equal"); ax.set_xlabel("x (A)"); ax.set_ylabel("y (A)")
ax.set_title(r"Zr HCP basal plane: $\mathbf{a}_1,\mathbf{a}_2$ at $120^\circ$")
ax.grid(alpha=0.3)
fig.tight_layout()


## Summary and where to go next

- A **`Phase`** is the crystal: `Lattice` + `UnitCell` + space group + `SymmetrySpec`.
- The **metric tensor** encodes all cell geometry; $V=\sqrt{\det\mathbf G}$ and the
  **reciprocal metric** gives exact $d$-spacings for every crystal system from one formula.
- PyTex's **fixture corpus** is provenance-backed and its imports are validated against
  declared site counts; phases **round-trip through CIF**.

**Next:** [Notebook 08](08_diffraction_geometry_and_kinematic_spots.ipynb) turns these
$d$-spacings into Bragg angles and reciprocal-lattice vectors, and
[Notebook 11](11_powder_xrd_workflows.ipynb) / [Notebook 12](12_saed_workflows.ipynb)
simulate full powder and SAED patterns from the very phases loaded here.